# P08 Agent Skills の仕組み ― 何がいつ文脈に入るか

> **この教材に登場する会社・人物・製品・数値・文書はすべて架空のものです。実在の企業・団体・個人とは関係ありません。**

講座の 2 つのスキル (月次レポート・契約書の一次レビュー) の SKILL.md を読み、決まりに照らし、段階 1・2・3 で文脈に入る量を測ります。

- 上から順に実行します。0 のセルは準備です (Colab でも手元の Jupyter でも同じ)。
- このノートブックは Claude を呼びません。API キーが無くても全部動きます。
- 🔴 データはすべて架空の会社・文書です。

## 0. 準備

In [ ]:
# 0-1. 実行場所を見分けて、キットのフォルダに移る (Colab のときだけ取得とインストールをする)
import os, sys, subprocess
REPO_URL = "https://github.com/gucchi123/mcp-skills-colab.git"
try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False
if IN_COLAB:
    KIT = "/content/mcp-skills-kit"
    if not os.path.exists(KIT):
        subprocess.run(["git", "clone", "-q", REPO_URL, KIT], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", f"{KIT}/requirements.txt"], check=True)
else:  # 手元: notebooks/ から開いた前提。キットの根 (common/ がある所) を探す
    KIT = os.path.abspath(".." if os.path.exists("../common/host.py") else ".")
os.chdir(KIT)
sys.path.insert(0, KIT)
print("実行場所:", "Colab" if IN_COLAB else "手元の Jupyter", "/ キットの根:", os.path.basename(KIT))

In [ ]:
# 0-2. このノートブックは Claude を呼びません (API キー不要)。ラボのモジュールを読めるようにする
from common.nb import safe
from common.kit import use_labs, utf8_console
use_labs(); utf8_console()

## 1. SKILL.md を読んで決まりに照らす
frontmatter (name・description・allowed-tools…) を読み、Agent Skills の決まり (name の文字・フォルダ名との一致・description の長さなど) を 1 つずつ確かめます。

In [ ]:
from run_mechanics import amain as mech
with safe():
    r = await mech(["--only", "solaha-monthly-report", "--show"])

## 2. わざと壊した見本を同じ検査に通す
大文字と _ の入った name、長すぎる description、frontmatter の無いファイル。どこで止まるかを見ます。

In [ ]:
with safe():
    r = await mech(["--part", "bad"])

## 3. 段階ごとに文脈に入る量
段階 1 (name と description) はいつも、段階 2 (本文) は使うときだけ、段階 3 (references・scripts) は要るときだけ。スクリプトは実際に走らせ、文脈に入るのは出力だけであることを確かめます。トークンは見積りです。

In [ ]:
with safe():
    r = await mech(["--part", "levels", "--show"])

## 4. スキルが 2 個のときと 20 個のとき
同じ中身を、Skill にする / システムプロンプトに全部書く / 道具の説明に全部書く、の 3 通りで、毎回の依頼に乗る量を比べます (20 個は名前だけ替えた写し)。

In [ ]:
with safe():
    r = await mech(["--part", "compare", "--n", "2,20"])

## 5. 何をどこに置くか
手順と型は Skill、データと操作は MCP、どの仕事にも効く方針はシステムプロンプト。2 つのスキルで分けてみます (allowed-tools の道具がサーバに在るかも確かめます)。

In [ ]:
with safe():
    r = await mech(["--part", "split"])